# 01 — Alberta Carbon Sequestration Agreement Source Comparison

## Objective

Compare the publicly downloadable Alberta Carbon Sequestration Agreement
shapefile against the NRCan-provided copy.

Determine whether they are:

1. byte-identical;
2. the same underlying dataset with downstream modifications; or
3. materially different source products.

The comparison should examine:

- file structure and names;
- feature counts;
- schema and field differences;
- CRS;
- geometry types and bounds;
- agreement and tract identifiers;
- shared attribute values;
- geometry equality;
- metadata and processing history.

In [4]:
import geopandas as gpd
import pandas as pd

from pathlib import Path

In [5]:
AER_DIR = Path(
    r"C:\Users\aviga\Research\potential data\Storage\AER\CS_agreements"
)

NRCAN_DIR = Path(
    r"C:\Users\aviga\Research\potential data\Storage\AB_NRCAN"
)

print("AER exists:", AER_DIR.exists())
print("NRCan exists:", NRCAN_DIR.exists())

AER exists: True
NRCan exists: True


Discover the shapefiles instead of hard-coding names:

In [6]:
aer_shps = sorted(AER_DIR.glob("*.shp"))
nrcan_shps = sorted(NRCAN_DIR.glob("*.shp"))

print("AER shapefiles:")
for path in aer_shps:
    print("  ", path.name)

print("\nNRCan shapefiles:")
for path in nrcan_shps:
    print("  ", path.name)

AER shapefiles:
   CS_Agreements.shp

NRCan shapefiles:
   ABCarbonSequestrationAgreement.shp


Load both shp files.

In [7]:
AER_SHP = AER_DIR / "CS_Agreements.shp"
NRCAN_SHP = NRCAN_DIR / "ABCarbonSequestrationAgreement.shp"

aer_gdf = gpd.read_file(AER_SHP)
nrcan_gdf = gpd.read_file(NRCAN_SHP)

Summary of shp files

In [9]:
summary = pd.DataFrame(
    {
        "AER": [
            len(aer_gdf),
            aer_gdf.crs.to_string() if aer_gdf.crs else None,
            sorted(
                aer_gdf.geometry.geom_type
                .dropna()
                .unique()
                .tolist()
            ),
            int(aer_gdf.geometry.isna().sum()),
            len(aer_gdf.columns),
        ],
        "NRCan": [
            len(nrcan_gdf),
            nrcan_gdf.crs.to_string() if nrcan_gdf.crs else None,
            sorted(
                nrcan_gdf.geometry.geom_type
                .dropna()
                .unique()
                .tolist()
            ),
            int(nrcan_gdf.geometry.isna().sum()),
            len(nrcan_gdf.columns),
        ],
    },
    index=[
        "feature_count",
        "crs",
        "geometry_types",
        "null_geometries",
        "column_count",
    ],
)

summary

,AER,NRCan
feature_count,45,34
crs,EPSG:3400,EPSG:3400
geometry_types,"[MultiPolygon, Polygon]","[MultiPolygon, Polygon]"
null_geometries,0,1
column_count,17,19


These are not the same shp files. But AER seems to potentially have more.

Exact field differences

In [10]:
aer_cols = set(aer_gdf.columns)
nrcan_cols = set(nrcan_gdf.columns)

print("AER-only fields:")
for col in sorted(aer_cols - nrcan_cols):
    print(f"  {col}")

print("\nNRCan-only fields:")
for col in sorted(nrcan_cols - aer_cols):
    print(f"  {col}")

print("\nShared fields:")
for col in sorted(aer_cols & nrcan_cols):
    print(f"  {col}")

AER-only fields:
  AgreementN
  AgreementT
  CUREXPIRY
  DesRep
  Shape_STAr
  Shape_STLe
  Status
  Tract
  Vintage
  ZoneDesc

NRCan-only fields:
  AGREENO
  AGREETYPE
  CUREXPTXT
  DESREP
  Latitude
  Longitude
  OBJECTID_1
  Phase
  STATUS
  TRACT
  VINTAGE
  ZONE

Shared fields:
  AGGROUP
  AGREEAREA
  CONTDATE
  MINTYPE
  ORGAREA
  TERMDATE
  geometry


In [15]:
## Normalize comparable AER field names to the NRCan naming convention for source-to-source comparison.

aer_compare = aer_gdf.rename(
    columns={
        "AgreementN": "AGREENO",
        "AgreementT": "AGREETYPE",
        "CUREXPIRY": "CUREXPTXT",
        "DesRep": "DESREP",
        "Status": "STATUS",
        "Tract": "TRACT",
        "Vintage": "VINTAGE",
        "ZoneDesc": "ZONE",
    }
).copy()

## Define the shared source attributes we expect to be semantically comparable after renaming.

compare_cols = [
    "AGREENO",
    "AGREETYPE",
    "TRACT",
    "AGGROUP",
    "MINTYPE",
    "STATUS",
    "DESREP",
    "ZONE",
    "ORGAREA",
    "AGREEAREA",
    "TERMDATE",
    "CONTDATE",
    "CUREXPTXT",
    "VINTAGE",
]

for col in compare_cols:
    print(
        f"{col:<12} "
        f"AER={col in aer_compare.columns}  "
        f"NRCan={col in nrcan_gdf.columns}"
    )

AGREENO      AER=True  NRCan=True
AGREETYPE    AER=True  NRCan=True
TRACT        AER=True  NRCan=True
AGGROUP      AER=True  NRCan=True
MINTYPE      AER=True  NRCan=True
STATUS       AER=True  NRCan=True
DESREP       AER=True  NRCan=True
ZONE         AER=True  NRCan=True
ORGAREA      AER=True  NRCan=True
AGREEAREA    AER=True  NRCan=True
TERMDATE     AER=True  NRCan=True
CONTDATE     AER=True  NRCan=True
CUREXPTXT    AER=True  NRCan=True
VINTAGE      AER=True  NRCan=True


In [16]:
## Define the agreement/tract source keys and compare record overlap between the two datasets.

key_cols = ["AGREENO", "TRACT"]

aer_keys = set(
    aer_compare[key_cols]
    .fillna("<NULL>")
    .astype(str)
    .apply(tuple, axis=1)
)

nrcan_keys = set(
    nrcan_gdf[key_cols]
    .fillna("<NULL>")
    .astype(str)
    .apply(tuple, axis=1)
)

print("AER keys:", len(aer_keys))
print("NRCan keys:", len(nrcan_keys))
print("Shared keys:", len(aer_keys & nrcan_keys))
print("AER-only keys:", len(aer_keys - nrcan_keys))
print("NRCan-only keys:", len(nrcan_keys - aer_keys))

AER keys: 45
NRCan keys: 34
Shared keys: 20
AER-only keys: 25
NRCan-only keys: 14


In [17]:
## Inspect agreement/tract keys that exist in only one source to understand how the datasets diverge.

aer_only_keys = sorted(aer_keys - nrcan_keys)
nrcan_only_keys = sorted(nrcan_keys - aer_keys)

print("AER-only keys:")
for key in aer_only_keys:
    print(key)

print("\nNRCan-only keys:")
for key in nrcan_only_keys:
    print(key)

AER-only keys:
('240148301', '00')
('250085901', '00')
('250108401', '00')
('260043401', '00')
('260088401', '00')
('260116601', '00')
('260127901', '00')
('5822100007', '00')
('5822100008', '00')
('5924060001', '00')
('5924070002', '00')
('5924090003', '01')
('5924090003', '02')
('5924090004', '01')
('5924090004', '02')
('5925070187', '00')
('5925100107', '00')
('5926040251', '00')
('6124070195', '00')
('6124090129', '00')
('6124090130', '00')
('6125050167', '00')
('6125060159', '00')
('6126020063', '00')
('6126060125', '00')

NRCan-only keys:
('5822100007', '01')
('5822100007', '02')
('5822100008', '01')
('5822100008', '02')
('5822100012', '00')
('5822120002', '00')
('5822120003', '01')
('5822120003', '02')
('5822120004', '00')
('5822120006', '00')
('5822120007', '00')
('5822120010', '00')
('5822120015', '00')
('<NULL>', '<NULL>')


In [18]:
## Compare agreement-number overlap separately from tract structure to identify renumbering versus tract changes.

aer_agreements = set(
    aer_compare["AGREENO"]
    .dropna()
    .astype(str)
)

nrcan_agreements = set(
    nrcan_gdf["AGREENO"]
    .dropna()
    .astype(str)
)

print("AER agreements:", len(aer_agreements))
print("NRCan agreements:", len(nrcan_agreements))
print("Shared agreements:", len(aer_agreements & nrcan_agreements))
print("AER-only agreements:", len(aer_agreements - nrcan_agreements))
print("NRCan-only agreements:", len(nrcan_agreements - aer_agreements))

AER agreements: 43
NRCan agreements: 30
Shared agreements: 22
AER-only agreements: 21
NRCan-only agreements: 8


In [19]:
## Inspect agreement numbers present in only one source to identify likely expired, removed, or newly added agreements.

aer_only_agreements = sorted(aer_agreements - nrcan_agreements)
nrcan_only_agreements = sorted(nrcan_agreements - aer_agreements)

print("AER-only agreements:")
for agreement in aer_only_agreements:
    print(agreement)

print("\nNRCan-only agreements:")
for agreement in nrcan_only_agreements:
    print(agreement)

AER-only agreements:
240148301
250085901
250108401
260043401
260088401
260116601
260127901
5924060001
5924070002
5924090003
5924090004
5925070187
5925100107
5926040251
6124070195
6124090129
6124090130
6125050167
6125060159
6126020063
6126060125

NRCan-only agreements:
5822100012
5822120002
5822120003
5822120004
5822120006
5822120007
5822120010
5822120015


In [20]:
## Compare date fields for agreements unique to each source to test whether the differences reflect temporal turnover.

date_cols = ["AGREENO", "TERMDATE", "CONTDATE", "CUREXPTXT", "STATUS"]

display(
    aer_compare[
        aer_compare["AGREENO"].astype(str).isin(aer_only_agreements)
    ][date_cols]
    .sort_values("AGREENO")
)

display(
    nrcan_gdf[
        nrcan_gdf["AGREENO"].astype(str).isin(nrcan_only_agreements)
    ][date_cols]
    .sort_values("AGREENO")
)

,AGREENO,TERMDATE,CONTDATE,CUREXPTXT,STATUS
43,240148301,2024/07/02,None,NaN,ACTIVE
30,250085901,2025/05/29,None,NaN,ACTIVE
38,250108401,2025/06/30,None,NaN,ACTIVE
37,260043401,2026/03/03,None,NaN,ACTIVE
40,260088401,2026/05/07,None,NaN,ACTIVE
42,260116601,2026/06/09,None,NaN,ACTIVE
44,260127901,2026/06/29,None,NaN,ACTIVE
22,5924060001,2024/06/25,None,2039/06/25,ACTIVE
24,5924070002,2024/07/30,None,2039/07/30,ACTIVE
26,5924090003,2024/09/18,None,2039/09/18,ACTIVE


,AGREENO,TERMDATE,CONTDATE,CUREXPTXT,STATUS
9,5822100012,2022/10/01,None,2027/10/01,ACTIVE
28,5822120002,2022/12/01,None,2027/12/01,ACTIVE
15,5822120003,2022/12/01,None,2027/12/01,ACTIVE
16,5822120003,2022/12/01,None,2027/12/01,ACTIVE
1,5822120004,2022/12/01,None,2027/12/01,ACTIVE
26,5822120006,2022/12/01,None,2027/12/01,ACTIVE
4,5822120007,2022/12/01,None,2027/12/01,ACTIVE
3,5822120010,2022/12/01,None,2027/12/01,ACTIVE
11,5822120015,2022/12/01,None,2027/12/01,ACTIVE


In [21]:
## Compare shared agreements across both datasets to test whether common records retain the same core attributes.

shared_agreements = sorted(aer_agreements & nrcan_agreements)

shared_compare_cols = [
    "AGREENO",
    "AGREETYPE",
    "AGGROUP",
    "MINTYPE",
    "STATUS",
    "DESREP",
    "ORGAREA",
    "AGREEAREA",
    "TERMDATE",
    "CONTDATE",
    "CUREXPTXT",
    "VINTAGE",
]

aer_shared = (
    aer_compare[
        aer_compare["AGREENO"].astype(str).isin(shared_agreements)
    ][shared_compare_cols]
    .drop_duplicates()
    .sort_values("AGREENO")
)

nrcan_shared = (
    nrcan_gdf[
        nrcan_gdf["AGREENO"].astype(str).isin(shared_agreements)
    ][shared_compare_cols]
    .drop_duplicates()
    .sort_values("AGREENO")
)

print("AER shared rows:", len(aer_shared))
print("NRCan shared rows:", len(nrcan_shared))

AER shared rows: 22
NRCan shared rows: 22


In [22]:
## Join the 22 shared agreements and identify which core attributes changed between the NRCan and AER versions.

shared_merged = aer_shared.merge(
    nrcan_shared,
    on="AGREENO",
    how="inner",
    suffixes=("_AER", "_NRCAN"),
)

comparison_fields = [
    "AGREETYPE",
    "AGGROUP",
    "MINTYPE",
    "STATUS",
    "DESREP",
    "ORGAREA",
    "AGREEAREA",
    "TERMDATE",
    "CONTDATE",
    "CUREXPTXT",
    "VINTAGE",
]

for field in comparison_fields:
    same = (
        shared_merged[f"{field}_AER"]
        .fillna("<NULL>")
        .astype(str)
        ==
        shared_merged[f"{field}_NRCAN"]
        .fillna("<NULL>")
        .astype(str)
    )

    print(
        f"{field:<12} "
        f"same={same.sum():>2}  "
        f"different={(~same).sum():>2}"
    )

AGREETYPE    same=22  different= 0
AGGROUP      same= 0  different=22
MINTYPE      same= 0  different=22
STATUS       same=22  different= 0
DESREP       same=18  different= 4
ORGAREA      same=22  different= 0
AGREEAREA    same=16  different= 6
TERMDATE     same=22  different= 0
CONTDATE     same=22  different= 0
CUREXPTXT    same=16  different= 6
VINTAGE      same=16  different= 6


In [23]:
## Inspect paired values for fields that differ systematically or partially between AER and NRCan.

fields_to_inspect = [
    "AGGROUP",
    "MINTYPE",
    "DESREP",
    "AGREEAREA",
    "CUREXPTXT",
    "VINTAGE",
]

for field in fields_to_inspect:
    pairs = (
        shared_merged[
            ["AGREENO", f"{field}_AER", f"{field}_NRCAN"]
        ]
        .drop_duplicates()
    )

    print(f"\n--- {field} ---")
    display(pairs)


--- AGGROUP ---


,AGREENO,AGGROUP_AER,AGGROUP_NRCAN
0,5822100007,PERMIT,AGREEMENT
1,5822100008,PERMIT,AGREEMENT
2,5822100009,PERMIT,AGREEMENT
3,5822100010,PERMIT,AGREEMENT
4,5822100011,PERMIT,AGREEMENT
5,5822120001,PERMIT,AGREEMENT
6,5822120005,PERMIT,AGREEMENT
7,5822120008,PERMIT,AGREEMENT
8,5822120009,PERMIT,AGREEMENT
9,5822120011,PERMIT,AGREEMENT



--- MINTYPE ---


,AGREENO,MINTYPE_AER,MINTYPE_NRCAN
0,5822100007,PORE SPACE,OTHER
1,5822100008,PORE SPACE,OTHER
2,5822100009,PORE SPACE,OTHER
3,5822100010,PORE SPACE,OTHER
4,5822100011,PORE SPACE,OTHER
5,5822120001,PORE SPACE,OTHER
6,5822120005,PORE SPACE,OTHER
7,5822120008,PORE SPACE,OTHER
8,5822120009,PORE SPACE,OTHER
9,5822120011,PORE SPACE,OTHER



--- DESREP ---


...

,AGREENO,AGREEAREA_AER,AGREEAREA_NRCAN
0,5822100007,140976.026,209711.482
1,5822100008,52480.000,536441.426
2,5822100009,55040.000,70400.000
3,5822100010,607996.600,946246.890
4,5822100011,947303.700,1024615.700
5,5822120001,73280.000,73280.000
6,5822120005,978624.000,978624.000
7,5822120008,66816.000,66816.000
8,5822120009,93952.000,93952.000
9,5822120011,1735872.000,1735872.000



--- CUREXPTXT ---


,AGREENO,CUREXPTXT_AER,CUREXPTXT_NRCAN
0,5822100007,2027/10/01,2027/10/01
1,5822100008,2027/10/01,2027/10/01
2,5822100009,2027/10/01,2027/10/01
3,5822100010,2027/10/01,2027/10/01
4,5822100011,2027/10/01,2027/10/01
5,5822120001,2027/12/01,2027/12/01
6,5822120005,2027/12/01,2027/12/01
7,5822120008,2027/12/01,2027/12/01
8,5822120009,2027/12/01,2027/12/01
9,5822120011,2027/12/01,2027/12/01



--- VINTAGE ---


,AGREENO,VINTAGE_AER,VINTAGE_NRCAN
0,5822100007,NaN,None
1,5822100008,NaN,None
2,5822100009,NaN,None
3,5822100010,NaN,None
4,5822100011,NaN,None
5,5822120001,NaN,None
6,5822120005,NaN,None
7,5822120008,NaN,None
8,5822120009,NaN,None
9,5822120011,NaN,None


In [24]:
## Inspect agreements whose current agreement area changed between the NRCan and AER snapshots.

area_changes = shared_merged[
    shared_merged["AGREEAREA_AER"] != shared_merged["AGREEAREA_NRCAN"]
][
    ["AGREENO", "AGREEAREA_NRCAN", "AGREEAREA_AER"]
].copy()

area_changes["AREA_CHANGE"] = (
    area_changes["AGREEAREA_AER"]
    - area_changes["AGREEAREA_NRCAN"]
)

area_changes

,AGREENO,AGREEAREA_NRCAN,AGREEAREA_AER,AREA_CHANGE
0,5822100007,209711.482,140976.026,-68735.456
1,5822100008,536441.426,52480.000,-483961.426
2,5822100009,70400.000,55040.000,-15360.000
3,5822100010,946246.890,607996.600,-338250.290
4,5822100011,1024615.700,947303.700,-77312.000
14,5822120018,127104.000,70144.000,-56960.000


In [25]:
## Compare geometry area for the six shared agreements whose reported agreement area changed.

changed_agreements = area_changes["AGREENO"].astype(str).tolist()

aer_geom_area = (
    aer_compare[
        aer_compare["AGREENO"].astype(str).isin(changed_agreements)
    ][["AGREENO", "geometry"]]
    .copy()
)

nrcan_geom_area = (
    nrcan_gdf[
        nrcan_gdf["AGREENO"].astype(str).isin(changed_agreements)
    ][["AGREENO", "geometry"]]
    .copy()
)

aer_geom_area["GEOM_AREA_AER_HA"] = aer_geom_area.geometry.area / 10_000
nrcan_geom_area["GEOM_AREA_NRCAN_HA"] = nrcan_geom_area.geometry.area / 10_000

geom_area_compare = (
    aer_geom_area[["AGREENO", "GEOM_AREA_AER_HA"]]
    .merge(
        nrcan_geom_area[["AGREENO", "GEOM_AREA_NRCAN_HA"]],
        on="AGREENO",
        how="inner",
    )
)

geom_area_compare

,AGREENO,GEOM_AREA_AER_HA,GEOM_AREA_NRCAN_HA
0,5822100007,142194.861431,1.421949e+05
1,5822100007,142194.861431,6.951972e+04
2,5822100008,53019.498498,5.419604e+05
3,5822100008,53019.498498,1.279144e+02
4,5822100009,55576.285362,7.102224e+04
5,5822100010,614183.690491,9.545537e+05
6,5822120018,70287.000957,1.274520e+05
7,5822100011,955972.605159,1.034155e+06


In [26]:
## Dissolve both datasets to one geometry per agreement before comparing shared agreement extents.

aer_dissolved = (
    aer_compare[
        aer_compare["AGREENO"].astype(str).isin(shared_agreements)
    ]
    .dissolve(by="AGREENO")
)

nrcan_dissolved = (
    nrcan_gdf[
        nrcan_gdf["AGREENO"].astype(str).isin(shared_agreements)
    ]
    .dissolve(by="AGREENO")
)

print("AER dissolved agreements:", len(aer_dissolved))
print("NRCan dissolved agreements:", len(nrcan_dissolved))

AER dissolved agreements: 22
NRCan dissolved agreements: 22


In [27]:
## Compare dissolved agreement geometry areas between AER and NRCan for the 22 shared agreements.

geom_compare = pd.DataFrame(
    {
        "GEOM_AREA_AER_HA": aer_dissolved.geometry.area / 10_000,
        "GEOM_AREA_NRCAN_HA": nrcan_dissolved.geometry.area / 10_000,
    }
)

geom_compare["AREA_CHANGE_HA"] = (
    geom_compare["GEOM_AREA_AER_HA"]
    - geom_compare["GEOM_AREA_NRCAN_HA"]
)

geom_compare["ABS_CHANGE_HA"] = geom_compare["AREA_CHANGE_HA"].abs()

geom_compare.sort_values(
    "ABS_CHANGE_HA",
    ascending=False,
)

,GEOM_AREA_AER_HA,GEOM_AREA_NRCAN_HA,AREA_CHANGE_HA,ABS_CHANGE_HA
AGREENO,,,,
5822100008,5.301950e+04,5.420883e+05,-4.890688e+05,4.890688e+05
5822100010,6.141837e+05,9.545537e+05,-3.403700e+05,3.403700e+05
5822100011,9.559726e+05,1.034155e+06,-7.818286e+04,7.818286e+04
5822100007,1.421949e+05,2.117146e+05,-6.951972e+04,6.951972e+04
5822120018,7.028700e+04,1.274520e+05,-5.716501e+04,5.716501e+04
5822100009,5.557629e+04,7.102224e+04,-1.544595e+04,1.544595e+04
5822120011,1.754089e+06,1.754089e+06,-4.494120e-05,4.494120e-05
5822120005,9.903040e+05,9.903040e+05,1.293665e-05,1.293665e-05
5822120016,1.356931e+05,1.356931e+05,-3.213092e-06,3.213092e-06


In [28]:
## Compare reported agreement-area changes against calculated polygon-area changes for the six materially revised agreements.

area_vs_geometry = area_changes.merge(
    geom_compare.reset_index(),
    on="AGREENO",
    how="inner",
)

area_vs_geometry[
    [
        "AGREENO",
        "AGREEAREA_NRCAN",
        "AGREEAREA_AER",
        "AREA_CHANGE",
        "AREA_CHANGE_HA",
    ]
]

,AGREENO,AGREEAREA_NRCAN,AGREEAREA_AER,AREA_CHANGE,AREA_CHANGE_HA
0,5822100007,209711.482,140976.026,-68735.456,-69519.718051
1,5822100008,536441.426,52480.000,-483961.426,-489068.835811
2,5822100009,70400.000,55040.000,-15360.000,-15445.954721
3,5822100010,946246.890,607996.600,-338250.290,-340370.018213
4,5822100011,1024615.700,947303.700,-77312.000,-78182.859491
5,5822120018,127104.000,70144.000,-56960.000,-57165.005764


In [29]:
## Quantify the difference between reported agreement-area changes and geometry-derived area changes.

area_vs_geometry["CHANGE_DIFFERENCE_HA"] = (
    area_vs_geometry["AREA_CHANGE_HA"]
    - area_vs_geometry["AREA_CHANGE"]
)

area_vs_geometry[
    [
        "AGREENO",
        "AREA_CHANGE",
        "AREA_CHANGE_HA",
        "CHANGE_DIFFERENCE_HA",
    ]
]

,AGREENO,AREA_CHANGE,AREA_CHANGE_HA,CHANGE_DIFFERENCE_HA
0,5822100007,-68735.456,-69519.718051,-784.262051
1,5822100008,-483961.426,-489068.835811,-5107.409811
2,5822100009,-15360.000,-15445.954721,-85.954721
3,5822100010,-338250.290,-340370.018213,-2119.728213
4,5822100011,-77312.000,-78182.859491,-870.859491
5,5822120018,-56960.000,-57165.005764,-205.005764


## Conclusion

The public AER `CS_Agreements.shp` dataset and the NRCan-provided
`ABCarbonSequestrationAgreement.shp` dataset are not identical files or simple
renamed copies.

The comparison indicates that they are different temporal snapshots of the
same Alberta carbon sequestration agreement dataset lineage.

Key findings:

- both use EPSG:3400 and contain Polygon/MultiPolygon agreement geometries;
- 22 agreement numbers are shared between the two datasets;
- the current AER release contains 21 agreements not present in the NRCan copy;
- the NRCan copy contains 8 older agreements not present in the current AER release;
- several shared agreements have changed tract structures;
- core identifiers and several administrative attributes remain consistent;
- some fields have been systematically recoded or enriched in the newer AER release;
- six shared agreements have materially revised agreement areas and polygon geometries;
- reported agreement-area changes closely track geometry-derived area changes.

For the CanCO₂Re storage pipeline, the publicly downloadable AER dataset will
therefore be treated as the authoritative reproducible bronze source. The
NRCan-provided copy will be retained only as a historical comparison dataset
and will not be required by the production pipeline.